In [1]:

# imports
import os
import sys
import types
import json
import base64

# figure size/format
fig_width = 7
fig_height = 5
fig_format = 'retina'
fig_dpi = 96
interactivity = ''
is_shiny = False
is_dashboard = False
plotly_connected = True

# matplotlib defaults / format
try:
  import matplotlib.pyplot as plt
  plt.rcParams['figure.figsize'] = (fig_width, fig_height)
  plt.rcParams['figure.dpi'] = fig_dpi
  plt.rcParams['savefig.dpi'] = "figure"

  # IPython 7.14 deprecated set_matplotlib_formats from IPython
  try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
  except ImportError:
    # Fall back to deprecated location for older IPython versions
    from IPython.display import set_matplotlib_formats
    
  set_matplotlib_formats(fig_format)
except Exception:
  pass

# plotly use connected mode
try:
  import plotly.io as pio
  if plotly_connected:
    pio.renderers.default = "notebook_connected"
  else:
    pio.renderers.default = "notebook"
  for template in pio.templates.keys():
    pio.templates[template].layout.margin = dict(t=30,r=0,b=0,l=0)
except Exception:
  pass

# disable itables paging for dashboards
if is_dashboard:
  try:
    from itables import options
    options.dom = 'fiBrtlp'
    options.maxBytes = 1024 * 1024
    options.language = dict(info = "Showing _TOTAL_ entries")
    options.classes = "display nowrap compact"
    options.paging = False
    options.searching = True
    options.ordering = True
    options.info = True
    options.lengthChange = False
    options.autoWidth = False
    options.responsive = True
    options.keys = True
    options.buttons = []
  except Exception:
    pass
  
  try:
    import altair as alt
    # By default, dashboards will have container sized
    # vega visualizations which allows them to flow reasonably
    theme_sentinel = '_quarto-dashboard-internal'
    def make_theme(name):
        nonTheme = alt.themes._plugins[name]    
        def patch_theme(*args, **kwargs):
            existingTheme = nonTheme()
            if 'height' not in existingTheme:
              existingTheme['height'] = 'container'
            if 'width' not in existingTheme:
              existingTheme['width'] = 'container'

            if 'config' not in existingTheme:
              existingTheme['config'] = dict()
            
            # Configure the default font sizes
            title_font_size = 15
            header_font_size = 13
            axis_font_size = 12
            legend_font_size = 12
            mark_font_size = 12
            tooltip = False

            config = existingTheme['config']

            # The Axis
            if 'axis' not in config:
              config['axis'] = dict()
            axis = config['axis']
            if 'labelFontSize' not in axis:
              axis['labelFontSize'] = axis_font_size
            if 'titleFontSize' not in axis:
              axis['titleFontSize'] = axis_font_size  

            # The legend
            if 'legend' not in config:
              config['legend'] = dict()
            legend = config['legend']
            if 'labelFontSize' not in legend:
              legend['labelFontSize'] = legend_font_size
            if 'titleFontSize' not in legend:
              legend['titleFontSize'] = legend_font_size  

            # The header
            if 'header' not in config:
              config['header'] = dict()
            header = config['header']
            if 'labelFontSize' not in header:
              header['labelFontSize'] = header_font_size
            if 'titleFontSize' not in header:
              header['titleFontSize'] = header_font_size    

            # Title
            if 'title' not in config:
              config['title'] = dict()
            title = config['title']
            if 'fontSize' not in title:
              title['fontSize'] = title_font_size

            # Marks
            if 'mark' not in config:
              config['mark'] = dict()
            mark = config['mark']
            if 'fontSize' not in mark:
              mark['fontSize'] = mark_font_size

            # Mark tooltips
            if tooltip and 'tooltip' not in mark:
              mark['tooltip'] = dict(content="encoding")

            return existingTheme
            
        return patch_theme

    # We can only do this once per session
    if theme_sentinel not in alt.themes.names():
      for name in alt.themes.names():
        alt.themes.register(name, make_theme(name))
      
      # register a sentinel theme so we only do this once
      alt.themes.register(theme_sentinel, make_theme('default'))
      alt.themes.enable('default')

  except Exception:
    pass

# enable pandas latex repr when targeting pdfs
try:
  import pandas as pd
  if fig_format == 'pdf':
    pd.set_option('display.latex.repr', True)
except Exception:
  pass

# interactivity
if interactivity:
  from IPython.core.interactiveshell import InteractiveShell
  InteractiveShell.ast_node_interactivity = interactivity

# NOTE: the kernel_deps code is repeated in the cleanup.py file
# (we can't easily share this code b/c of the way it is run).
# If you edit this code also edit the same code in cleanup.py!

# output kernel dependencies
kernel_deps = dict()
for module in list(sys.modules.values()):
  # Some modules play games with sys.modules (e.g. email/__init__.py
  # in the standard library), and occasionally this can cause strange
  # failures in getattr.  Just ignore anything that's not an ordinary
  # module.
  if not isinstance(module, types.ModuleType):
    continue
  path = getattr(module, "__file__", None)
  if not path:
    continue
  if path.endswith(".pyc") or path.endswith(".pyo"):
    path = path[:-1]
  if not os.path.exists(path):
    continue
  kernel_deps[path] = os.stat(path).st_mtime
print(json.dumps(kernel_deps))

# set run_path if requested
run_path = 'L1VzZXJzL3JjYXAvd29yay9zZXNzaW9uaXphdGlvbg=='
if run_path:
  # hex-decode the path
  run_path = base64.b64decode(run_path.encode("utf-8")).decode("utf-8")
  os.chdir(run_path)

# reset state
%reset

# shiny
# Checking for shiny by using False directly because we're after the %reset. We don't want
# to set a variable that stays in global scope.
if False:
  try:
    import htmltools as _htmltools
    import ast as _ast

    _htmltools.html_dependency_render_mode = "json"

    # This decorator will be added to all function definitions
    def _display_if_has_repr_html(x):
      try:
        # IPython 7.14 preferred import
        from IPython.display import display, HTML
      except:
        from IPython.core.display import display, HTML

      if hasattr(x, '_repr_html_'):
        display(HTML(x._repr_html_()))
      return x

    # ideally we would undo the call to ast_transformers.append
    # at the end of this block whenver an error occurs, we do 
    # this for now as it will only be a problem if the user 
    # switches from shiny to not-shiny mode (and even then likely
    # won't matter)
    import builtins
    builtins._display_if_has_repr_html = _display_if_has_repr_html

    class _FunctionDefReprHtml(_ast.NodeTransformer):
      def visit_FunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

      def visit_AsyncFunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

    ip = get_ipython()
    ip.ast_transformers.append(_FunctionDefReprHtml())

  except:
    pass

def ojs_define(**kwargs):
  import json
  try:
    # IPython 7.14 preferred import
    from IPython.display import display, HTML
  except:
    from IPython.core.display import display, HTML

  # do some minor magic for convenience when handling pandas
  # dataframes
  def convert(v):
    try:
      import pandas as pd
    except ModuleNotFoundError: # don't do the magic when pandas is not available
      return v
    if type(v) == pd.Series:
      v = pd.DataFrame(v)
    if type(v) == pd.DataFrame:
      j = json.loads(v.T.to_json(orient='split'))
      return dict((k,v) for (k,v) in zip(j["index"], j["data"]))
    else:
      return v

  v = dict(contents=list(dict(name=key, value=convert(value)) for (key, value) in kwargs.items()))
  display(HTML('<script type="ojs-define">' + json.dumps(v) + '</script>'), metadata=dict(ojs_define = True))
globals()["ojs_define"] = ojs_define
globals()["__spec__"] = None

{"/Users/rcap/.local/share/uv/python/cpython-3.14.0-macos-aarch64-none/lib/python3.14/importlib/_bootstrap.py": 1765295500.4805598, "/Users/rcap/.local/share/uv/python/cpython-3.14.0-macos-aarch64-none/lib/python3.14/importlib/_bootstrap_external.py": 1765295500.4807293, "/Users/rcap/.local/share/uv/python/cpython-3.14.0-macos-aarch64-none/lib/python3.14/zipimport.py": 1765295500.561649, "/Users/rcap/.local/share/uv/python/cpython-3.14.0-macos-aarch64-none/lib/python3.14/codecs.py": 1765295500.3791692, "/Users/rcap/.local/share/uv/python/cpython-3.14.0-macos-aarch64-none/lib/python3.14/encodings/aliases.py": 1765295500.3898857, "/Users/rcap/.local/share/uv/python/cpython-3.14.0-macos-aarch64-none/lib/python3.14/encodings/__init__.py": 1765295500.389467, "/Users/rcap/.local/share/uv/python/cpython-3.14.0-macos-aarch64-none/lib/python3.14/encodings/utf_8.py": 1765295500.4589815, "/Users/rcap/.local/share/uv/python/cpython-3.14.0-macos-aarch64-none/lib/python3.14/abc.py": 1765295500.37455

In [2]:
from skrub import TableReport
import polars as pl

df = pl.scan_parquet("data/wowah_data_raw.parquet")
# we collect the dataframe because the TableReport does not accept LazyFrames
TableReport(df.collect())

,,,,,,


In [3]:
def make_data(df):
    """
    Add a timestamp column with monthly resolution, the first month for each
    character, and a constant column that acts as flag. 
    """

    data = (
        df.with_columns(
            pl.col("timestamp").dt.truncate("1mo").alias("month"),
            first_month=pl.col("timestamp").dt.truncate("1mo").min().over("char"),
        )
        .unique(subset=["char", "month"])
        .with_columns(pl.lit(True).alias("has_played"))
    )
    return data
data = make_data(df)

In [4]:
import polars as pl
df1 = pl.DataFrame({"col": [1,2,3]})
df2 = pl.DataFrame({"col": ["a","b"]})
df1.join(df2, how="cross")

col,col_right
i64,str
1,"""a"""
1,"""b"""
2,"""a"""
2,"""b"""
3,"""a"""
3,"""b"""


In [5]:
def make_user_month(df):
    """
    Create a DataFrame with all unique combinations of users and months.
    
    The month range is extracted from the dataset.  
    """

    months = pl.datetime_range(
        start=df.select(pl.col("timestamp").dt.truncate("1mo").min()).collect().item(),
        end=df.select(pl.col("timestamp").dt.truncate("1mo").max()).collect().item(),
        interval="1mo",
        closed="both",
        eager=True,
    )

    char_month = (
        df
        .select("char")
        .unique()
        .join(months.to_frame(name="month").lazy(), how="cross")
    )
    return char_month

user_month = make_user_month(df)
user_month.collect()

char,month
i32,datetime[μs]
84555,2008-01-01 00:00:00
84555,2008-02-01 00:00:00
84555,2008-03-01 00:00:00
84555,2008-04-01 00:00:00
84555,2008-05-01 00:00:00
…,…
74450,2008-08-01 00:00:00
74450,2008-09-01 00:00:00
74450,2008-10-01 00:00:00


In [6]:
def add_churn(user_month, data):
    """
    Merge the user-month DataFrame with the actual play data.
    The resulting DataFrame will have a "has_played" column indicating
    whether the user played in that month. Any missing values in "has_played" are
    filled with False: missing values mean that the player did not play in that
    particular month.
    """

    df_with_user_month = (
        user_month.join(
            data,
            on=["char", "month"],
            how="left",
        )
        .with_columns(pl.col("has_played").fill_null(False))
        # Fill missing values in the first_month column with each character's
        # first month
        .with_columns(first_month=pl.col("first_month").min().over("char"))
        # Remove (char, month) rows where the month precedes the character's
        # first month
        .filter(pl.col("first_month") <= pl.col("month"))
        .select(
            "char",
            "month",
            "has_played",
            "first_month",
        )
    )
    return df_with_user_month


r = add_churn(user_month, data)
r.collect()

char,month,has_played,first_month
i32,datetime[μs],bool,datetime[μs]
89423,2008-12-01 00:00:00,true,2008-12-01 00:00:00
75781,2008-07-01 00:00:00,true,2008-07-01 00:00:00
75781,2008-08-01 00:00:00,false,2008-07-01 00:00:00
75781,2008-09-01 00:00:00,false,2008-07-01 00:00:00
75781,2008-10-01 00:00:00,false,2008-07-01 00:00:00
…,…,…,…
63818,2008-08-01 00:00:00,true,2008-01-01 00:00:00
63818,2008-09-01 00:00:00,false,2008-01-01 00:00:00
63818,2008-10-01 00:00:00,false,2008-01-01 00:00:00


In [7]:
def build_churn_dataset(historical_data):
    # Load the dataset
    data = make_data(historical_data)
    # Add the combinations of user months
    user_month = make_user_month(historical_data)
    churn_data = add_churn(user_month, data)
    return churn_data

churn_data = build_churn_dataset(df)
churn_data.collect()

char,month,has_played,first_month
i32,datetime[μs],bool,datetime[μs]
79729,2008-10-01 00:00:00,true,2008-10-01 00:00:00
79729,2008-11-01 00:00:00,false,2008-10-01 00:00:00
79729,2008-12-01 00:00:00,false,2008-10-01 00:00:00
79205,2008-10-01 00:00:00,true,2008-10-01 00:00:00
79205,2008-11-01 00:00:00,false,2008-10-01 00:00:00
…,…,…,…
24214,2008-08-01 00:00:00,true,2008-02-01 00:00:00
24214,2008-09-01 00:00:00,false,2008-02-01 00:00:00
24214,2008-10-01 00:00:00,true,2008-02-01 00:00:00


In [8]:
churn_data.group_by("month").agg(pl.len()).sort("len").collect()

month,len
datetime[μs],u32
2008-01-01 00:00:00,8600
2008-02-01 00:00:00,11692
2008-03-01 00:00:00,14209
2008-04-01 00:00:00,15902
2008-05-01 00:00:00,17384
…,…
2008-08-01 00:00:00,22853
2008-09-01 00:00:00,24434
2008-10-01 00:00:00,31175


In [9]:
from datetime import datetime

# We define the boundaries of the experiment manually
MIN_DATE = datetime.strptime("2008-01-01", "%Y-%m-%d")
MAX_DATE = datetime.strptime("2008-12-30", "%Y-%m-%d")

class Splitter:
    """
    This splitter iterates over the months. It selects all the months up to the 
    split point, which is our cutoff point: it's when we want to perform some 
    operation on users that are marked as "churn risk", i.e., have been predicted
    to churn by the model. 
    """
    def split(self, user_month, has_played=None):
        """
        user_month is our "X", while has_played is the target variable we want 
        to predict
        """
        # has_played is not needed by this splitter, we are only splitting by month
        del has_played
        time_range = pl.date_range(MIN_DATE, MAX_DATE, "1mo", eager=True)
        for split_point in time_range:
            # The test month is the month *after* the split point
            test_month = pl.Series([split_point]).dt.offset_by("1mo").first()
            # Train indices are up to split_point
            # We want a one-month gap
            train_idx = (
                user_month.with_row_index("idx")
                .filter(pl.col("month") <= split_point)["idx"]
                .to_list()
            )
            # Test indices are for the month after split_point
            test_idx = (
                user_month.with_row_index("idx")
                .filter(pl.col("month") == test_month)["idx"]
                .to_list()
            )
            if train_idx and test_idx:
                print(f"Working on month {split_point}")
                yield train_idx, test_idx

    def get_n_splits(self, X, y):
        return sum(1 for _ in self.split(X, y))

In [10]:
# %%
# This function is needed to make sure that we are only ever using historical data
# up to the given month - 1 month. This is to avoid any leakage in the data.
def add_features(X, historical_data, session_gap=30, use_location=True, add_gini=False):
    features_by_month = []

    # Create a session encoder with a 30 minute timeout
    # This encoder is used as a stateless transformer so it is refitted for every
    # month
    session_encoder = SessionEncoder(
        split_by="char", timestamp_col="timestamp", session_gap=session_gap
    )
    historical_data = historical_data.with_columns(
        month=pl.col("timestamp").dt.truncate("1mo")
    )
    last_month = X["month"].max()

    # Grouping by character and zone so that I can get the time spent in each zone
    # Even if users leave the zone, this lets me find how much time a user spends in
    # a given zone
    session_encoder_zone = SessionEncoder(
        split_by=["char", "zone"], timestamp_col="timestamp", session_gap=session_gap
    )
    # Adding fixed features: these features are fixed by character so they don't
    # change over time.
    # historical_data is selected up until the last month because if I select only
    # a single month then any character that did not play in that month will be
    # missing those features
    X = X.join(
        historical_data.filter(pl.col("month") <= last_month)
        .select("char", "race", "charclass")
        .unique("char"),
        on="char",
        how="left",
        maintain_order="left",
    ).with_row_index()  # adding row index so that I can reorder at the end after
    # concatenating
    # kinda defeats the point of using data ops but I think it simplifies the code

    # This is used to add the historical data up to the given month
    # Sorting months is not needed, but forces a consistent order (better for debugging)
    for month in X["month"].unique().sort():
        this_month_X = filter_df_by_month(X, month)

        # Selecting only the entries in the historical data whose month + 2 is equal
        # to the month I am trying to predict on.
        # This means that if the "target month" is April, then the historical data
        # should be filtered to keep only the rows where the current month + 2
        # is equal to April, that is, the month is February. This is equivalent
        # to saying "I want the rows for the current month - 2 months", but it's
        # easier to implement
        kept_historical_data = historical_data.with_columns(
            pl.col("month").dt.offset_by("2mo")
        ).filter(pl.col("month") == month)

        # Session features: a session starts from a heartbeat, then it ends when
        # no more heartbeats are detected for session_gap minutes
        historical_data_with_sessions = session_encoder.fit_transform(
            kept_historical_data
        )
        historical_data_with_sessions = get_session_duration(
            historical_data_with_sessions
        )

        # General features: add session based and playerbase features
        df_with_features = add_general_features(
            this_month_X, historical_data_with_sessions
        )

        # Location features can be useful but take much longer to generate
        if use_location:
            # Zone-session features: a session lasts from the first time a character
            # enters a zone to the moment it leaves it
            # This is useful to get zone-specific features
            historical_data_zone_sessions = session_encoder_zone.fit_transform(
                kept_historical_data
            )
            historical_data_zone_sessions = get_session_duration(
                historical_data_zone_sessions
            )
            df_with_features = add_location_features(
                df_with_features,
                historical_data_zone_sessions,
                add_gini=add_gini,
            )

        features_by_month.append(df_with_features)
        assert len(df_with_features) == len(this_month_X)

    X_res = pl.concat(features_by_month, how="vertical")
    X_res = X_res.sort("index").drop("index")

    return X_res

In [11]:
def sample_by_char(df, fraction=0.1):
    """
    Sample a fraction of chars and return all their data.
    """
    data = df.filter(
        pl.col("char").is_in(
            df.select(pl.col("char").unique())
            .sample(fraction=fraction, seed=42)["char"]
            .implode()
        )
    )
    return data


def load(file, fraction=0.1):
    if fraction == 1:
        print("Returning all users")
        return pl.scan_parquet(file)

    print(f"Sampling {fraction * 100}% of the users")
    df = pl.scan_parquet(file)
    df = sample_by_char(df.collect(), fraction=fraction)
    return df.lazy()

In [12]:
# %%a
import skrub

def make_data_op():
    historical_data_file = skrub.var("historical_data_file")
    historical_data = historical_data_file.skb.apply_func(load)
    # In the guild column, null values are denoted by -1. This step replaces -1
    # with None. 
    historical_data = historical_data.with_columns(
        pl.when(pl.col("guild") == -1)
        .then(None)
        .otherwise(pl.col("guild"))
        .alias("guild")
    )

    user_month_has_played = historical_data.skb.apply_func(build_churn_dataset)
    X = user_month_has_played["char", "month"].skb.mark_as_X(cv=Splitter())
    y = user_month_has_played["has_played"].skb.mark_as_y()

    # Hyperparameters
    session_gap = skrub.choose_from([60], name="session_gap")
    use_location = skrub.choose_bool(name="location_features")
    add_gini = skrub.choose_bool(name="add_gini")

    all_features = X.skb.apply_func(
        add_features,
        historical_data.collect(),
        session_gap=session_gap,
        use_location=use_location,
        add_gini=add_gini
    )
    encoded = all_features.skb.apply(skrub.TableVectorizer())
    # data_op = encoded.skb.apply(SimpleImputer()).skb.apply(LogisticRegression(), y=y)
    data_op = encoded.skb.apply(
        HGB(learning_rate=skrub.choose_float(0.01, 1.0, log=True)), y=y
    )
    # data_op = encoded.skb.apply(DummyClassifier(), y=y)
    return data_op
